<a href="https://colab.research.google.com/github/vineetsalar88/CompilerDesign/blob/master/21marchAutoEncoderKNNResnet18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
pip install torch torchvision scikit-learn tqdm

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

import numpy as np
from sklearn.cluster import KMeans
from tqdm import tqdm

In [ ]:
data_path = "/content/drive/MyDrive/ResearchData/March26/AnnonatedDataset2March26"   # unlabeled images
batch_size = 32
latent_dim = 128
epochs = 10
num_clusters = 6

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data/
   dummy/
      img1.jpg
      img2.jpg

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

dataset = ImageFolder(data_path, transform=transform)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64*32*32, latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64*32*32),
            nn.ReLU(),
            nn.Unflatten(1, (64,32,32)),
            nn.ConvTranspose2d(64,32,3,stride=2,padding=1,output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32,3,3,stride=2,padding=1,output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

In [ ]:
ae = Autoencoder().to(device)
optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for epoch in range(epochs):
    ae.train()

    for images, _ in tqdm(loader):
        images = images.to(device)

        optimizer.zero_grad()
        recon, _ = ae(images)

        loss = criterion(recon, images)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

100%|██████████| 35/35 [02:22<00:00,  4.06s/it]


Epoch 1, Loss: 0.02603062614798546


100%|██████████| 35/35 [01:11<00:00,  2.05s/it]


Epoch 2, Loss: 0.025125406682491302


100%|██████████| 35/35 [01:11<00:00,  2.05s/it]


...

Epoch 8, Loss: 0.011913090944290161


100%|██████████| 35/35 [01:13<00:00,  2.10s/it]


Epoch 9, Loss: 0.01081471424549818


100%|██████████| 35/35 [01:13<00:00,  2.10s/it]

Epoch 10, Loss: 0.010686223395168781


In [ ]:
features = []

ae.eval()
with torch.no_grad():
    for images, _ in loader:
        images = images.to(device)
        _, z = ae(images)

        features.append(z.cpu().numpy())

features = np.vstack(features)

# Normalize
from sklearn.preprocessing import normalize
features = normalize(features)

# PCA (ADD HERE)
from sklearn.decomposition import PCA
features = PCA(n_components=64).fit_transform(features)

# K-Means
kmeans = KMeans(n_clusters=num_clusters, n_init=10)
cluster_labels = kmeans.fit_predict(features)

In [ ]:
print("Cluster labels assigned:", len(cluster_labels))

Cluster labels assigned: 1115


In [ ]:
class ClusterDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, labels):
        self.dataset = dataset
        self.labels = labels

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        label = int(self.labels[idx])
        return img, label

In [ ]:
cluster_dataset = ClusterDataset(dataset, cluster_labels)
cluster_loader = DataLoader(cluster_dataset, batch_size=32, shuffle=True)

In [ ]:
import torchvision.models as models

cnn = models.resnet18(pretrained=True)
cnn.fc = nn.Linear(cnn.fc.in_features, num_clusters)
cnn = cnn.to(device)

optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    cnn.train()

    for images, labels in cluster_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = cnn(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    print(f"CNN Epoch {epoch+1}, Loss: {loss.item()}")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 184MB/s]


CNN Epoch 1, Loss: 2.0185999870300293
CNN Epoch 2, Loss: 0.8980849981307983
CNN Epoch 3, Loss: 0.4612661302089691
CNN Epoch 4, Loss: 0.20395991206169128
CNN Epoch 5, Loss: 0.1224227249622345


In [ ]:
image_paths = [sample[0] for sample in dataset.samples]

In [ ]:
import os

output_dir = "/content/drive/MyDrive/Researchdata/clustered_data"

for i in range(num_clusters):
    os.makedirs(os.path.join(output_dir, f"cluster_{i}"), exist_ok=True)

In [ ]:
import shutil

for img_path, label in zip(image_paths, cluster_labels):

    cluster_folder = os.path.join(output_dir, f"cluster_{label}")

    filename = os.path.basename(img_path)

    shutil.copy(img_path, os.path.join(cluster_folder, filename))

In [ ]:
loader = DataLoader(dataset, shuffle=False)

In [ ]:
new_name = f"{label}_{i}.jpg"